# AOL9 / AO2609 零信号诊断

## tl;dr

- 17,017 根正式回测对应 `SHFE.AO2609.5m`，不是 `SHFE.AOL9.5m`。
- 该样本有完整缠论结构，但 45 个最终有效背驰全部属于盘整背驰；策略只接受已确认、标准的一买/一卖，因此交易信号为 0。
- 当前 69,289 根 `SHFE.AOL9.5m` 全量回测实际有 1 个标准一买、1 个开多信号和 1 次成交；由于没有反向标准一卖，已平仓交易数仍为 0。
- `trade_count` 是已平仓交易数，不等于信号数。

## Context & Methods

诊断对象是仓库内不可变 comparison/run 产物及其内容寻址的共享缠论运行时。检查粒度为数据集 revision、正式 run、最终有效语义对象和执行阶段。

### Key Assumptions

- `runtime.pkl` 是本项目在 `data_root` 下生成并以 SHA-256 校验的可信本地缓存。
- 最终对象计数按 `(object_type, object_id)` 应用 upsert/delete 后计算；原始 upsert 次数包含同一对象的因果修订，不能直接当作独立信号数。
- `trades.parquet` 只记录已经完成配对的交易。

## Data

### 1. Load exact manifests and trusted runtime artifacts

In [ ]:
from pathlib import Path
import collections
import json
import pickle
import sys

import pyarrow.parquet as pq

workspace = Path.cwd()
if not (workspace / 'python' / 'src').exists():
    workspace = workspace.parent
sys.path.insert(0, str(workspace / 'python' / 'src'))
data_root = workspace / 'trading-data'

comparison_17k = 'comparison-20260826T155130000000000-3053e7a5029c61a2534e'
comparison_aol9 = 'comparison-20260827T133620000000000-14d72060fc030dad785e'
run_17k = 'job-20260826T155130000000000-564d68c3aa62b0436cd4'
run_aol9 = 'job-20260827T133620000000000-a591312e70ae7fbcfacb'

def read_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))

manifest_17k = read_json(data_root / 'comparisons' / comparison_17k / 'comparison.json')
manifest_aol9 = read_json(data_root / 'comparisons' / comparison_aol9 / 'comparison.json')
print('17k dataset:', manifest_17k['dataset'], manifest_17k['range'])
print('AOL9 dataset:', manifest_aol9['dataset'], manifest_aol9['range'])

### 2. Check source coverage and ingestion quality

In [ ]:
def dataset_meta(dataset_id: str, revision: str):
    return read_json(data_root / 'normalized' / dataset_id / revision.removeprefix('sha256:')[:12] / 'meta.json')

for manifest in [manifest_17k, manifest_aol9]:
    identity = manifest['dataset']
    meta = dataset_meta(identity['dataset_id'], identity['data_revision'])
    print(json.dumps({
        'dataset_id': meta['dataset_id'],
        'source_path': meta['source']['path'],
        'coverage': meta['coverage'],
        'quality': meta['quality'],
    }, ensure_ascii=False, indent=2))

## Results

### 3. Reconstruct the final structural funnel

In [ ]:
def final_live_objects(dependency_id: str):
    path = data_root / 'cache' / 'comparison_dependencies' / dependency_id / 'runtime.pkl'
    cache = pickle.loads(path.read_bytes())
    runtime = next(iter(cache.values()))[0]
    live = {}
    for event in runtime.emitter.events:
        key = (event.object_type, event.object_id)
        if event.operation == 'delete':
            live.pop(key, None)
        elif event.operation == 'upsert':
            live[key] = json.loads(event.payload_json)
    return runtime, live

def funnel(manifest):
    dependency_id = manifest['shared_dependency']['dependency_id']
    runtime, live = final_live_objects(dependency_id)
    counts = collections.Counter(kind for kind, _ in live)
    divergences = [payload for (kind, _), payload in live.items() if kind == 'divergence']
    points = [payload for (kind, _), payload in live.items() if kind == 'trade_point']
    return {
        'raw_bars': len(runtime.raw_bars),
        'included_bars': len(runtime.included),
        'segments': counts['segment'],
        'segment_centres': counts['segment_zhongshu'],
        'divergences': counts['divergence'],
        'divergence_kinds': dict(collections.Counter(p.get('divergence_kind') for p in divergences)),
        'trade_points': counts['trade_point'],
        'trade_point_types': dict(collections.Counter(f"{p.get('signal_type')}|{p.get('signal_class')}" for p in points)),
    }

funnel_17k = funnel(manifest_17k)
funnel_aol9 = funnel(manifest_aol9)
print('17k funnel:', json.dumps(funnel_17k, ensure_ascii=False, indent=2, default=str))
print('AOL9 funnel:', json.dumps(funnel_aol9, ensure_ascii=False, indent=2, default=str))

### 4. Reconcile signal, risk, order, fill, and completed-trade counts

In [ ]:
def run_stage_counts(run_id: str):
    directory = data_root / 'runs' / run_id
    names = ['strategy_states', 'stage_signals', 'trade_signals', 'risk_decisions', 'orders', 'fills', 'trades', 'positions']
    result = {}
    for name in names:
        path = directory / f'{name}.parquet'
        result[name] = pq.ParquetFile(path).metadata.num_rows
    result['summary'] = read_json(directory / 'summary.json')
    if result['positions']:
        result['final_position'] = pq.read_table(directory / 'positions.parquet').to_pylist()[-1]
    return result

for run_id in [run_17k, run_aol9]:
    print(run_id)
    print(json.dumps(run_stage_counts(run_id), ensure_ascii=False, indent=2))

## Takeaways

1. 17,017 根样本的 0 信号发生在策略过滤之前：权威结构层没有生成标准、已确认的一买/一卖。风险层不是原因。
2. 该样本并不缺结构；盘整背驰、类买卖点和标准三买/三卖都存在，只是不属于 `trend_divergence_reversal` 的入场集合。
3. 17,017 根数据有 1,224 根零成交量和 365 个质量缺口警告，值得单独评估其对结构频率的影响，但这些警告不能解释成程序漏掉已经存在的标准一买/一卖。
4. 当前 AOL9 全量的 0 `trade_count` 是口径问题：实际已有一个开多信号和成交，只是未平仓。界面应同时展示交易信号数、成交数、已平仓交易数和期末持仓。
5. 一键比较回测与单周期可靠性研究是两个独立任务；后者不会自动继承前者选中的策略。